In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

DRIVE_DIR = "/content/drive/MyDrive/hajj_real_data"
os.makedirs(DRIVE_DIR, exist_ok=True)

IMAGES_DIR = os.path.join(DRIVE_DIR, "images")
os.makedirs(IMAGES_DIR, exist_ok=True)

CSV_PATH = os.path.join(DRIVE_DIR, "real_data.csv")

print(f"[OK] drive connected")
print(f"[OK] csv path: {CSV_PATH}")
print(f"[OK] images path: {IMAGES_DIR}")


In [ ]:
import os
import json
import pandas as pd

# 1. Setup paths
DRIVE_DIR = "/content/drive/MyDrive/hajj_real_data"
IMAGES_DIR = os.path.join(DRIVE_DIR, "images")
CSV_PATH = os.path.join(DRIVE_DIR, "real_data.csv")
JSON_PATH = "/content/bad_ads_ids.json"

print("Starting cleaning process...")

# 2. Read the bad IDs
if not os.path.exists(JSON_PATH):
    print("Error: bad_ads_ids.json file not found. Ensure it is uploaded to Colab.")
else:
    with open(JSON_PATH, "r") as f:
        bad_ids = set(json.load(f))

    print(f"Found {len(bad_ids)} ads that will be completely deleted.")

    # 3. Clean the CSV
    if os.path.exists(CSV_PATH):
        df = pd.read_csv(CSV_PATH)
        original_len = len(df)

        # Filter out rows where ID is in the bad IDs list
        df_clean = df[~df["Sample ID"].isin(bad_ids)]
        cleaned_len = len(df_clean)

        # Save the cleaned CSV, overwriting the old one
        df_clean.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
        print(f"CSV cleaned successfully! Reduced from {original_len} to {cleaned_len} valid ads.")
    else:
        print("Warning: CSV file not found at the specified path.")

    # 4. Delete bad images from Drive to save space
    deleted_images = 0
    not_found_images = 0

    for ad_id in bad_ids:
        img_path = os.path.join(IMAGES_DIR, f"{ad_id}.jpg")
        if os.path.exists(img_path):
            os.remove(img_path)
            deleted_images += 1
        else:
            not_found_images += 1

    print(f"Deleted {deleted_images} images from Drive permanently.")
    if not_found_images > 0:
        print(f"Notice: {not_found_images} images were not found (already deleted or missing).")

    print("\nProcess completed successfully! The CSV and images are now 100% clean.")


In [ ]:
import pandas as pd
import os

# 1. Define base paths
DRIVE_DIR = "/content/drive/MyDrive"
REAL_DATA_CSV = os.path.join(DRIVE_DIR, "hajj_real_data", "real_data.csv")
FAKE_DATA_CSV = os.path.join(DRIVE_DIR, "hajj_fraud_dataset", "fake_data_2000.csv")

# New workspace directory for combined datasets
COMBINED_DIR = os.path.join(DRIVE_DIR, "hajj_processing_workspace")
os.makedirs(COMBINED_DIR, exist_ok=True)

print(" Loading Real Data...")
if not os.path.exists(REAL_DATA_CSV):
    print(" Error: Real data CSV not found at:", REAL_DATA_CSV)
else:
    df_real = pd.read_csv(REAL_DATA_CSV)
    real_count = len(df_real)
    print(f" Loaded {real_count} cleaned real ads.")

    print(" Loading Fake Data...")
    if not os.path.exists(FAKE_DATA_CSV):
        print(" Error: Fake data CSV not found at:", FAKE_DATA_CSV)
    else:
        df_fake = pd.read_csv(FAKE_DATA_CSV)
        print(f" Loaded {len(df_fake)} original fake ads.")

        # 2. Extract a random sample from Fake Ads matching the exact length of Real Ads
        if len(df_fake) >= real_count:
            # Using random_state=42 ensures reproducibility across multiple runs
            df_fake_balanced = df_fake.sample(n=real_count, random_state=42)
        else:
            print(" Warning: Not enough fake ads available! Using all available fake data.")
            df_fake_balanced = df_fake.copy()

        # 3. Save matching datasets to the new workspace directory
        NEW_REAL_CSV = os.path.join(COMBINED_DIR, "real_data_ready.csv")
        NEW_FAKE_CSV = os.path.join(COMBINED_DIR, "fake_data_ready.csv")

        df_real.to_csv(NEW_REAL_CSV, index=False, encoding="utf-8-sig")
        df_fake_balanced.to_csv(NEW_FAKE_CSV, index=False, encoding="utf-8-sig")

        print("\ Process Completed Successfully!")
        print(f" Workspace created at: {COMBINED_DIR}")
        print(f" Real Data Size: {real_count} ads saved to real_data_ready.csv")
        print(f" Fake Data Size: {len(df_fake_balanced)} ads saved to fake_data_ready.csv")
        print(" The dataset is now perfectly balanced (50/50) and ready for processing!")


In [ ]:
import pandas as pd
from IPython.display import display

# Define paths to the newly created datasets
DRIVE_DIR = "/content/drive/MyDrive/hajj_processing_workspace"
REAL_DATA_CSV = f"{DRIVE_DIR}/real_data_ready.csv"
FAKE_DATA_CSV = f"{DRIVE_DIR}/fake_data_ready.csv"

print("Loading datasets from Drive...")
df_real = pd.read_csv(REAL_DATA_CSV)
df_fake = pd.read_csv(FAKE_DATA_CSV)

print("\n============================================================")
print("REAL DATASET ANALYSIS")
print("============================================================")
print("\n[Preview] First 5 rows of Real Data:")
display(df_real.head())

print("\n[Breakdown] 'Advertisement Type' counts (Real Data):")
print(df_real['Advertisement Type'].value_counts(dropna=False))

print("\n============================================================")
print("FAKE DATASET ANALYSIS")
print("============================================================")
print("\n[Preview] First 5 rows of Fake Data:")
display(df_fake.head())

print("\n[Breakdown] 'Advertisement Type' counts (Fake Data):")
print(df_fake['Advertisement Type'].value_counts(dropna=False))


In [ ]:
import pandas as pd

# Define paths
DRIVE_DIR = "/content/drive/MyDrive/hajj_processing_workspace"
FAKE_DATA_CSV = f"{DRIVE_DIR}/fake_data_ready.csv"

# Load Fake Data
df_fake = pd.read_csv(FAKE_DATA_CSV)

# Mapping Function
def unify_ad_type(row):
    ad_type_val = str(row.get('Advertisement Type', ''))
    text_val = str(row.get('Text Content', ''))

    combined_text = (ad_type_val + " " + text_val).lower()

    if any(word in combined_text for word in ['حج', 'hajj']):
        return 'hajj_package'
    elif any(word in combined_text for word in ['عمر', 'umrah']):
        return 'umrah_package'
    elif any(word in combined_text for word in ['تأشير', 'تاشير', 'فيزا', 'visa']):
        return 'visa_service'
    elif any(word in combined_text for word in ['سكن', 'فندق', 'اقام', 'إقام', 'غرف', 'accommodation']):
        return 'accommodation'
    elif any(word in combined_text for word in ['مك', 'مدين', 'مزار', 'طواف', 'makkah']):
        return 'makkah_related'
    else:
        return 'islamic_travel'

print("Unifying Advertisement Types in Fake Data...")

# Apply the mapping column by column
df_fake['Advertisement Type'] = df_fake.apply(unify_ad_type, axis=1)

# Save the updated file
df_fake.to_csv(FAKE_DATA_CSV, index=False, encoding="utf-8-sig")

print("Done! Here is the new unified breakdown for Fake Data:\n")
print(df_fake['Advertisement Type'].value_counts(dropna=False))


In [ ]:
import pandas as pd
import os

DRIVE_DIR = "/content/drive/MyDrive/hajj_processing_workspace"
REAL_DATA_CSV = f"{DRIVE_DIR}/real_data_ready.csv"
FAKE_DATA_CSV = f"{DRIVE_DIR}/fake_data_ready.csv"

translation_dict = {
    "hajj_package": "باقة حج",
    "umrah_package": "باقة عمرة",
    "visa_service": "خدمات تأشيرات",
    "accommodation": "سكن وإقامة",
    "makkah_related": "متعلق بمستلزمات مكة والمزارات",
    "islamic_travel": "سياحة دينية عامة"
}

print("Loading datasets...")
df_real = pd.read_csv(REAL_DATA_CSV)
df_fake = pd.read_csv(FAKE_DATA_CSV)

print("Translating 'Advertisement Type' to Arabic...")

df_real['Advertisement Type'] = df_real['Advertisement Type'].map(translation_dict).fillna(df_real['Advertisement Type'])
df_fake['Advertisement Type'] = df_fake['Advertisement Type'].map(translation_dict).fillna(df_fake['Advertisement Type'])

df_real.to_csv(REAL_DATA_CSV, index=False, encoding="utf-8-sig")
df_fake.to_csv(FAKE_DATA_CSV, index=False, encoding="utf-8-sig")

print(" Translation Completed Successfully!\n")

print("=========================================")
print(" REAL DATA (Arabic Breakdown):")
print("=========================================")
print(df_real['Advertisement Type'].value_counts(dropna=False))

print("\n=========================================")
print(" FAKE DATA (Arabic Breakdown):")
print("=========================================")
print(df_fake['Advertisement Type'].value_counts(dropna=False))


In [ ]:
import pandas as pd
import os

WORKSPACE_DIR = "/content/drive/MyDrive/hajj_processing_workspace"
REAL_DATA_CSV = f"{WORKSPACE_DIR}/real_data_ready.csv"
FAKE_DATA_CSV = f"{WORKSPACE_DIR}/fake_data_ready.csv"

FINAL_DIR = "/content/drive/MyDrive/hajj_final_dataset"
os.makedirs(FINAL_DIR, exist_ok=True)

FINAL_REAL_CSV = f"{FINAL_DIR}/real_data_final.csv"
FINAL_FAKE_CSV = f"{FINAL_DIR}/fake_data_final.csv"

print("Loading datasets...")
df_real = pd.read_csv(REAL_DATA_CSV)
df_fake = pd.read_csv(FAKE_DATA_CSV)

print("Shuffling data heavily...")

df_real_shuffled = df_real.sample(frac=1, random_state=42).reset_index(drop=True)
df_fake_shuffled = df_fake.sample(frac=1, random_state=123).reset_index(drop=True)

print("Saving shuffled datasets to the final directory...")
df_real_shuffled.to_csv(FINAL_REAL_CSV, index=False, encoding="utf-8-sig")
df_fake_shuffled.to_csv(FINAL_FAKE_CSV, index=False, encoding="utf-8-sig")

print("Done.")
print("Real Data Shuffled and saved to: ", FINAL_REAL_CSV)
print("Fake Data Shuffled and saved to: ", FINAL_FAKE_CSV)
print("Ready for your final processing.")


In [ ]:
import pandas as pd
from IPython.display import display

FINAL_DIR = "/content/drive/MyDrive/hajj_final_dataset"
FINAL_REAL_CSV = f"{FINAL_DIR}/real_data_final.csv"
FINAL_FAKE_CSV = f"{FINAL_DIR}/fake_data_final.csv"

COMBINED_CSV = f"{FINAL_DIR}/combined_dataset_final.csv"

print("Loading individual shuffled datasets...")
df_real = pd.read_csv(FINAL_REAL_CSV)
df_fake = pd.read_csv(FINAL_FAKE_CSV)

print("Concatenating into one unified dataset...")
df_combined = pd.concat([df_real, df_fake], ignore_index=True)

print("Shuffling the combined dataset...")
df_combined = df_combined.sample(frac=1, random_state=777).reset_index(drop=True)

print("Saving combined dataset...")
df_combined.to_csv(COMBINED_CSV, index=False, encoding="utf-8-sig")

print("\nDone. First 5 rows of the unified dataset:")
display(df_combined.head())

print("\nGlobal Dataset Information:")
print("Total columns:", len(df_combined.columns))
print("Total rows:", len(df_combined))
print("Unified file saved successfully at:", COMBINED_CSV)


In [ ]:
display(df_combined.head(100))


In [ ]:
import pandas as pd

COMBINED_CSV = "/content/drive/MyDrive/hajj_final_dataset/combined_dataset_final.csv"
df = pd.read_csv(COMBINED_CSV)

print("============================================================")
print("FINAL DATASET STATISTICAL SUMMARY")
print("============================================================")

total_ads = len(df)
print(f"\n[1] Total Advertisements: {total_ads}")

print("\n[2] Label Distribution (Real vs Fake):")
if 'Label' in df.columns:
    print(df['Label'].value_counts())
else:
    print("Label column not found.")

print("\n[3] Advertisement Types Breakdown:")
if 'Advertisement Type' in df.columns:
    print(df['Advertisement Type'].value_counts())
else:
    print("Advertisement Type column not found.")

print("\n[4] Dialect Percentages:")
if 'Dialect' in df.columns:
    dialect_pct = df['Dialect'].value_counts(normalize=True) * 100
    for dialect, pct in dialect_pct.items():
        print(f"  {dialect}: {pct:.2f}%")
else:
    print("Dialect column not found.")

print("\n============================================================")
